# M22 · Detecting earthquakes: network coincidence, template matching and a deep-learning detector

**Week 3 · about 12 h** · before this: M20, M21, M30

> **한국어 요약** 상주 ML 3.9 지진 전후 세 시간의 연속 기록에서 지진을 찾습니다. 관측망 동시 트리거와 템플릿 매칭(정규화 교차상관, MAD 문턱값, 상대 규모)을 직접 구현하고, PhaseNet의 피킹을 여러분이 만든 연관 함수로 묶어 세 방법의 검출 목록을 기상청 목록과 비교합니다. 마지막으로 템플릿 사이의 중복 검출을 합치고 정직한 검증 계획을 씁니다.

**You will**

- Detect events by network coincidence and template matching
- Set a detection threshold from the MAD
- Compare templates, PhaseNet and the KMA catalog
- Merge repeated matches and plan an honest validation

**New words:** network coincidence (관측망 동시 트리거), template matching (템플릿 매칭), normalized cross-correlation (정규화 교차상관), network cross-correlation (관측망 합산 상관), detection threshold (검출 문턱값), relative magnitude (상대 규모), null test (귀무 검정 (널 테스트)), back-projected origin time (역투영 발생 시각) · [all terms](../../../docs/glossary.md)

**Optional reading:** Gibbons & Ringdal (2006), the method; Peng & Zhao (2009), the methods

Log any AI help in `work/ai_log/M22.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M22.yaml](../../../.sgtl/criteria/M22.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019_hours"              # your STP download of the three hours (M03)
EVENT_FOLDER = "20190721020418"           # its folder in the download (UTC origin time of the ML 3.9)
MAINSHOCK_ID = "2019009762"               # KMA id of the ML 3.9 of 21 July 2019 (first template)
SMALL_ID = "2019009764"                   # KMA id of the ML 1.5, 6.5 minutes later (second template)
M20_PICKS = "../picks/M20_manual_picks.csv"   # your picks of the ML 3.9 (M20, task 5)
SMALL_PICKS = "../picks/M22_ml15_picks.csv"   # your picks of the ML 1.5 (Part 5)
FMIN_HZ, FMAX_HZ, CORNERS = 2.0, 10.0, 4  # causal band-pass of every detector here (Hz, poles)
TAPER_S = 5.0                             # taper at each end of the records (s)
STA_S, LTA_S = 0.5, 10.0                  # Part 1: STA and LTA windows (s), as in M20
THR_ON, THR_OFF = 4.0, 1.5                # Part 1: trigger thresholds, as in M20
MIN_STATIONS = 3                          # Part 1: stations that must trigger together
EXTENSION_S = 5.0                         # Part 1: coincidence extension (s)
PRE_S, P_LEN_S, S_LEN_S = 0.2, 2.0, 3.0   # Part 5: window start before a pick; P and S window lengths (s)
K_MAD = 8.0                               # Parts 3, 5: threshold = median + K_MAD x MAD
MIN_GAP_S = 2.0                           # Parts 3, 5: of two detections closer than this (s), the higher stays
MATCH_S = 2.0                             # Part 5: a detection this close (s) to a KMA origin time is that event
DEDUP_WINDOW_S = 2.0                      # Part 7: merge window from a group's first match (s)
CALIBRATION_WINDOW_S = (60.0, 1260.0)      # Part 7: rehearsal window (s after T0; already seen)
EVALUATION_WINDOW_S = (7200.0, 10200.0)    # Part 7: separate rehearsal window (s after T0)
VALIDATION_GUARD_S = 60.0                 # Part 7: guard around templates, windows and record edges (s)
PN_WEIGHTS = ("stead", "2")               # Part 6: PhaseNet weights (name, version) of M21
PN_THRESHOLD = 0.3                        # Part 6: PhaseNet pick threshold for P and S
PN_THRESHOLD_CHANGED = 0.1                # Part 6: the one changed setting of the third run
VELOCITY_MODEL = "kim1983"                # Part 6: model of the travel times (your direct_time of M30)
ASSOC_WINDOW_S = 2.0                      # Part 6: association window (s)
ASSOC_MIN_STATIONS = 3                    # Part 6: stations an event needs
FIGURE_DIR = "figures"                    # your figures go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M22")

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime, read
from obspy.geodetics import gps2dist_azimuth

from sgtl_tutorial import synth_detect

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
Path(SMALL_PICKS).parent.mkdir(parents=True, exist_ok=True)
# SeisBench looks for its weights in this folder; 'bin/sgtl-install seisbench' put the course's weights there.
_ = os.environ.setdefault("SEISBENCH_CACHE_ROOT", str(st.pq.sgtl_home() / "seisbench"))

## Part 0 · Three hours of records (about 20 min)

A **continuous record** runs for hours, and nobody has told you what is in it. **Earthquake detection** finds the
earthquakes in it. `sangju2019_hours` holds three hours around the ML 3.9 Sangju mainshock of 21 July 2019, at
eight stations. Fetch it first in a terminal: `bin/sgtl data fetch sangju2019_hours`, again with `--apply`, then
`bin/sgtl data verify sangju2019_hours`.

In [ ]:
cat = st.data.kma_catalog()
main = cat.loc[cat["event_id"] == MAINSHOCK_ID].iloc[0]
EV_LAT, EV_LON, EV_DEPTH_KM = float(main["latitude"]), float(main["longitude"]), float(main["depth_km"])

SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / EVENT_FOLDER
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no records in {SAC_DIR}. Fetch the {DATASET} data with STP first (module "
                            "M03: sgtl data fetch sangju2019_hours), or set DATASET in the parameters cell.")
raw = Stream()
for band in ("HH", "EL"):
    for path in sorted((SAC_DIR / band).glob("*.sac")):
        raw += read(str(path))
raw.sort(keys=["network", "station", "channel"])
if len({tr.id for tr in raw}) != len(raw):
    raise ValueError("Some channel comes in more than one piece (a gap in the records); this module needs "
                     "unbroken records. Fetch the data again, or ask your mentor.")
T0 = min(tr.stats.starttime for tr in raw)
T1 = max(tr.stats.endtime for tr in raw)
DELTA = raw[0].stats.delta
print(f"{len(raw)} channels from {T0} to {T1} (UTC), {raw[0].stats.sampling_rate:g} samples per second")

rows = []
for tr in raw.select(component="Z"):
    epi_km, az, _ = gps2dist_azimuth(EV_LAT, EV_LON, tr.stats.sac.stla, tr.stats.sac.stlo)
    rows.append({"station": f"{tr.stats.network}.{tr.stats.station}", "band": tr.stats.channel[:2],
                 "epi_km": epi_km / 1000.0, "azimuth_deg": az, "lat": tr.stats.sac.stla, "lon": tr.stats.sac.stlo})
stations = pd.DataFrame(rows).sort_values("epi_km", ignore_index=True)
print(stations.round({"epi_km": 1, "azimuth_deg": 0}).to_string(index=False))

in_window = cat[(cat["origin_utc"] >= pd.Timestamp(T0.datetime, tz="UTC"))
                & (cat["origin_utc"] <= pd.Timestamp(T1.datetime, tz="UTC"))]
print(f"\nKMA catalog, {T0.strftime('%H:%M:%S')} to {T1.strftime('%H:%M:%S')} UTC:")
print(in_window[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))
KMA_TIMES = {str(r.event_id): UTCDateTime(r.origin_utc.isoformat()) for r in in_window.itertuples()}
KMA_ML = {str(r.event_id): float(r.ml) for r in in_window.itertuples()}

if not Path(M20_PICKS).is_file():
    raise FileNotFoundError(f"{M20_PICKS} is missing. The first template comes from your own picks of the ML 3.9: "
                            "finish task M20-T5 of module M20 first (its picker saves the file).")
m20_picks = pd.read_csv(M20_PICKS)
print(f"\nyour M20 picks: {len(m20_picks)} rows at {m20_picks['station'].nunique()} stations; "
      f"{sorted(set(m20_picks['station']) & set(stations['station']))} are in these records")

All detectors use the same prepared records: the mean removed, the ends **tapered**, and a **causal** band-pass
from `FMIN_HZ` to `FMAX_HZ` Hz. The figure shows the nearest station, with the KMA events marked.

In [ ]:
def prepare(stream):
    """A prepared copy: float64, mean removed, ends tapered over TAPER_S, causal band-pass FMIN_HZ-FMAX_HZ."""
    out = stream.copy()
    for tr in out:
        tr.data = tr.data.astype(np.float64)
    out.detrend("demean")
    out.taper(max_percentage=None, max_length=TAPER_S, type="hann")
    out.filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=CORNERS, zerophase=False)
    return out


prepared = prepare(raw)
nearest = stations.loc[0, "station"]
z_near = prepared.select(network=nearest.split(".")[0], station=nearest.split(".")[1], component="Z")[0]
fig, ax = plt.subplots(figsize=(8.0, 2.8))
ax.plot(z_near.times() / 60.0, z_near.data, color="k", lw=0.3)
for number, (event_id, when) in enumerate(KMA_TIMES.items()):
    ax.axvline((when - z_near.stats.starttime) / 60.0, color="tab:red", lw=0.8, ls="--")
    ax.text((when - z_near.stats.starttime) / 60.0, 1.0, f" ML {KMA_ML[event_id]:.1f} ",
            ha="right" if number % 2 == 0 else "left", color="tab:red", transform=ax.get_xaxis_transform(),
            va="bottom", fontsize=8)
ax.set_xlabel(f"Minutes after {T0.strftime('%H:%M:%S')} UTC")
ax.set_ylabel(f"{z_near.id} (counts)")
ax.set_ylim(-60.0 * np.std(z_near.data[:30000]), 60.0 * np.std(z_near.data[:30000]))
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "three_hours_nearest.png", dpi=150)
print(f"{z_near.id}: noise {np.std(z_near.data[:30000]):.0f} counts in the first 5 minutes (standard deviation)")

The y-axis is cut at 60 times the noise. Some small bursts may be earthquakes, others local noise: one station
alone cannot tell.

## Part 1 · Network coincidence (about 60 min)

Your STA/LTA trigger of M20 also fires on trucks and wind. An earthquake, though, reaches all nearby stations
within seconds. **Network coincidence** declares an event only when enough stations trigger together: their
triggers overlap, or one starts within a short **coincidence extension** after another ends. Chains count: A with
B and B with C make one event.

### M22-T1 · Network coincidence · core · about 60 min

Write `coincidence_events(triggers, min_stations, extension_s)`. Sort all triggers by start time. A trigger joins
the current event when it starts at most `extension_s` after the event's end, and the end grows. Otherwise it
starts a new event. Keep the events with at least `min_stations` stations.

Example (`min_stations = 3`, `extension_s = 1.5`):

- triggers `{"XX.STA1": [(10.0, 12.0)], "XX.STA2": [(11.0, 13.0), (40.0, 41.0)], "XX.STA3": [(14.0, 15.0)]}`
- give `[(10.0, 15.0, ("XX.STA1", "XX.STA2", "XX.STA3"))]`.

Not allowed: `obspy.signal.trigger` and `pyrocko`.

In [ ]:
# Before you code: with your M20 settings on three hours of eight stations, about how many network detections do
# you expect with 3 stations, and with 6?
st.predict("M22-T1", """

""")

In [ ]:
# Your sta_lta and trigger_windows of module M20 (its tasks 1 and 2): paste both here, unchanged. Part 1 runs them
# on the three hours after the check; the check of this task does not need them.
# YOUR CODE HERE
raise NotImplementedError("M22-T1: write your code here")

In [ ]:
def coincidence_events(triggers, min_stations, extension_s):
    """Network detections: chains of overlapping triggers (plus extension_s) with at least min_stations stations.

    triggers     : dict {station name: list of (on_s, off_s)}, seconds after one common start time; any station
                   may have an empty list
    min_stations : whole number (1 or more)
    extension_s  : seconds (0 or more)
    Put the triggers of all stations into one list sorted by on_s. A trigger joins the current event when its on_s
    is at most the event's end + extension_s (the edge counts); the end then becomes the later of the end and its
    off_s. Otherwise it starts a new event (start: its on_s, end: its off_s).
    returns      : list with one tuple (start_s, end_s, stations) per event with at least min_stations different
                   stations, in time order: two floats and a sorted tuple of the different station names; [] when
                   there is none
    Not allowed here: obspy.signal.trigger (its coincidence_trigger), the trigger method of a trace or a stream,
    and pyrocko.
    """
    # Step 1: one list of (on, off, station) of all stations, sorted by the on time.
    # Step 2: walk through it; a trigger joins the current event when on <= end + extension_s (the end may grow);
    #         otherwise it starts a new event.
    # Step 3: keep the events with at least min_stations different stations.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T1: write your code here")

In [ ]:
st.check("M22-T1", func=coincidence_events)

Now the real records: your M20 functions on every vertical channel, then your `coincidence_events` for 2 to 8
stations. A detection covers a KMA event when it starts at most 15 s after the origin time and ends after it.

In [ ]:
station_triggers = {}
for tr in prepared.select(component="Z"):
    rate = tr.stats.sampling_rate
    pairs = trigger_windows(sta_lta(tr.data, int(STA_S * rate), int(LTA_S * rate)), THR_ON, THR_OFF)
    shift = float(tr.stats.starttime - T0)
    station_triggers[f"{tr.stats.network}.{tr.stats.station}"] = [(shift + on / rate, shift + off / rate)
                                                                   for on, off in pairs]
print("triggers per station:", {name: len(v) for name, v in sorted(station_triggers.items())})

kma_seconds = {event_id: float(when - T0) for event_id, when in KMA_TIMES.items()}


def covers(start, end, t):
    """True when origin time t (s after T0) lies between 15 s before a detection's start and its end."""
    return start - 15.0 <= t <= end


coincidence_table = []
for needed in range(2, 9):
    found = coincidence_events(station_triggers, needed, EXTENSION_S)
    with_kma = sum(any(covers(start, end, t) for t in kma_seconds.values()) for start, end, _ in found)
    coincidence_table.append({"min_stations": needed, "network_detections": len(found),
                              "covering_a_kma_event": with_kma,
                              "per_hour": round(len(found) / ((T1 - T0) / 3600), 1)})
print(pd.DataFrame(coincidence_table).to_string(index=False))
network_found = coincidence_events(station_triggers, MIN_STATIONS, EXTENSION_S)
print(f"\nwith MIN_STATIONS = {MIN_STATIONS}: {len(network_found)} detections; those with 5 or more stations "
      "(start, start in s after T0, length, stations):")
for start, end, names in network_found:
    if len(names) >= 5:
        kma = [e for e, t in kma_seconds.items() if covers(start, end, t)]
        print(f"  {T0 + start}  {start:8.1f} s  {end - start:5.1f} s  {', '.join(names)}"
              + (f"  (KMA {kma[0]})" if kma else ""))

**Explain:** What does the table say about choosing `min_stations` for a network of eight stations?

## Part 2 · Template matching (about 60 min)

**Template matching** asks whether a piece of record looks like a known earthquake. The **template** is a short
piece of that earthquake's record, such as 2 s of its P wave on one channel. A small event close by sends a scaled
copy of it, even when hidden in the noise.

The likeness of template $t$ and the piece $x$ of a record that starts at sample $i$ is their **correlation
coefficient**:

$$r_i = \frac{\sum_k t'_k\, x'_{i+k}}{\sqrt{\sum_k t'^2_k}\,\sqrt{\sum_k x'^2_{i+k}}}$$

The primes mean "minus its own mean". $r$ lies between -1 and 1. It ignores size: a small copy in quiet noise
still gives nearly 1. One $r$ for every $i$ is the **normalized cross-correlation**.

### M22-T2 · A normalized cross-correlation · core · about 60 min

Write `normalized_cc(template, data)`: $r_i$ for every piece `data[i:i + m]`, with `m = len(template)`. It
returns `len(data) - m + 1` floats, 0 where a piece is flat.

Example: `template = [1, 2, 3]` and `data = [0, 1, 2, 3, 2, 4, 6]` give `[1.0, 1.0, 0.0, 0.5, 1.0]`.

Part 5 runs it on 1,080,000 samples about 80 times, so no Python loop over the pieces: use running totals
(`np.cumsum`) and `np.correlate`.

Not allowed: libraries that compute the coefficient for you (the docstring lists them).

In [ ]:
# Before you code: a record holds a copy of the template 10 times smaller than the noise. Will its coefficient be
# near 1, near 0.1 or near 0?
st.predict("M22-T2", """

""")

In [ ]:
def normalized_cc(template, data):
    """Correlation coefficient of the template with every piece data[i:i + m] (m = len(template)).

    template : 1-D sequence of m numbers (a list, or a numpy array of any number type, 32-bit integers included)
    data     : 1-D sequence of n >= m numbers (the same kinds)
    returns  : numpy float64 array of n - m + 1 values in [-1, 1]; value i is the coefficient of the template with
               data[i:i + m], both minus their own means; 0 where a piece of data is flat (variance 0; a variance
               that rounding made slightly negative counts as 0)
    Not allowed here: anything that gives a correlation coefficient or a cosine similarity (the coefficient
    without removing the means) for you: obspy.signal (its cross_correlation and trigger), numpy.corrcoef (also
    of numpy.ma), statistics.correlation, scipy.stats (pearsonr), scipy.spatial.distance, pandas' corr, corrwith
    and autocorr (also on rolling, expanding and ewm windows), statsmodels, scikit-learn, xarray, polars, dask,
    PyTorch, JAX, TensorFlow, Keras, CuPy, stumpy, mass_ts, tslearn, librosa, EQcorrscan, fast_matched_filter,
    scikit-image, OpenCV, pyrocko, PocketQuake and the course package.
    """
    # Step 1: float64 arrays; the template minus its mean (then sum_k t'_k x'_(i+k) = sum_k t'_k x_(i+k)).
    # Step 2: the numerator for every i: a sliding product of the template with the data.
    # Step 3: for every piece: sum x and sum x^2 over m samples from running totals; m times its variance is
    #         sum x^2 - (sum x)^2 / m.
    # Step 4: divide by sqrt(sum t'^2 * m * variance) where that is above 0; 0 elsewhere.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T2: write your code here")

In [ ]:
st.check("M22-T2", func=normalized_cc)

Your function on a made-up record of 60 s: noise with four copies of a made-up waveform. Two copies are smaller
than the noise, one is upside down.

In [ ]:
demo_rng = np.random.default_rng(22)
demo_wave = synth_detect.wavelet(demo_rng, 200)
demo_record, demo_truth = synth_detect.record_with_copies(
    demo_rng, demo_wave, 6000, starts=[700, 2100, 3600, 4900], scales=[5.0, 0.5, -2.0, 0.25], noise=1.0, smooth=3)
demo_cc = normalized_cc(demo_wave, demo_record)
for start, scale in zip(demo_truth["starts"], demo_truth["scales"]):
    print(f"copy at {start / 100:5.2f} s, {abs(scale) * np.abs(demo_wave).max() / np.std(demo_record):5.2f} "
          f"times the record's standard deviation{' (upside down)' if scale < 0 else ''}: "
          f"coefficient {demo_cc[start]:+.3f}")
away = np.ones(demo_cc.size, dtype=bool)
for start in demo_truth["starts"]:
    away[max(0, start - 200):start + 200] = False
print(f"largest value more than 2 s from every copy: {demo_cc[away].max():.3f}; "
      f"smallest: {demo_cc[away].min():.3f}")
fig, (ax_rec, ax_cc) = plt.subplots(2, 1, figsize=(7.5, 4.2), sharex=True)
ax_rec.plot(np.arange(demo_record.size) / 100.0, demo_record, color="k", lw=0.4)
ax_rec.set_ylabel("Made-up record")
ax_cc.plot(np.arange(demo_cc.size) / 100.0, demo_cc, color="k", lw=0.6)
ax_cc.set_ylabel("Correlation coefficient")
ax_cc.set_xlabel("Time of the piece's first sample (s); dashed: the copies")
for start in demo_truth["starts"]:
    for ax in (ax_rec, ax_cc):
        ax.axvline(start / 100.0, color="tab:red", lw=0.7, ls="--")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "normalized_cc_made_up.png", dpi=150)

**Explain:** Which copies stand out from the noise in the coefficient, and why do the others not?

## Part 3 · How high is high enough? (about 45 min)

One channel's coefficient wanders with the noise. An earthquake lines up on all channels once you allow for the
**moveout**: the wave reaches a farther station later. So shift each channel back by its lag (how long after the
origin time its window starts) and average them all: the **network cross-correlation**. Noise peaks do not line
up, so they average away.

A **detection threshold** decides which peaks count. Measure the noise with the **median absolute deviation**
(MAD, M21), which a few large earthquakes cannot pull up:

$$\mathrm{MAD} = \mathrm{median}\,|c_i - \mathrm{median}(c)|, \qquad \text{threshold} = \mathrm{median}(c) +
k\cdot\mathrm{MAD}$$

A peak above it that is no earthquake is a **false detection**; their number per hour is the **false-alarm
rate**. A larger $k$ gives fewer of them, and fewer small earthquakes.

### M22-T3 · A threshold from the MAD · core · about 45 min

Write `mad_detections(cc, delta, k, min_gap_s)`: the threshold above (use `np.median`), and as detections the
picks of your M21 `picks_from_probability(cc, delta, threshold, min_gap_s)`. Return `(times_s, values, threshold)`.

Example (`delta = 0.5`, `k = 4`, `min_gap_s = 1.0`):

- `cc = [0.0, 0.1, -0.1, 0.0, 0.9, 0.0, 0.05, -0.05, 0.6, 0.0]`
- gives the threshold 0.2 (median 0, MAD 0.05) and detections at 2.0 s (0.9) and 4.0 s (0.6).

Not allowed: ready-made MAD functions and peak finders (the docstring lists them).

In [ ]:
# Before you code: on one hour of pure noise, about how many false detections do you expect at k = 4, 6 and 8?
st.predict("M22-T3", """

""")

In [ ]:
# Your picks_from_probability of module M21 (its task 1): paste it here, unchanged.
# YOUR CODE HERE
raise NotImplementedError("M22-T3: write your code here")

In [ ]:
def mad_detections(cc, delta, k, min_gap_s):
    """Detections in a network cross-correlation: peaks at or above median + k * MAD, at least min_gap_s apart.

    cc        : 1-D numpy array, one value per sample; do not change it
    delta     : sample spacing (s)
    k         : the threshold in MADs above the median (> 0); MAD = median(|cc - median(cc)|) over all samples, the
                plain MAD (not scaled, not the standard deviation)
    min_gap_s : two detections closer than this (s) cannot both stay (the higher one stays); detections exactly
                min_gap_s apart both stay
    returns   : (times_s, values, threshold): the detection times in s after the first sample and the values of cc
                there (two 1-D float arrays sorted by time), and the threshold (a float)
    Not allowed here, also in your picks_from_probability: scipy.stats (median_abs_deviation), scipy.signal
    (find_peaks and its relatives), scipy.ndimage, statsmodels, astropy, pandas' mad, seisbench, obspy.signal,
    torch and the course package.
    """
    # Step 1: the median of cc, then the median of the distances |cc - median|: the MAD.
    # Step 2: threshold = median + k * MAD.
    # Step 3: the peaks of your picks_from_probability with that threshold and min_gap_s.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T3: write your code here")

In [ ]:
st.check("M22-T3", func=mad_detections)

Now the false-alarm rate: a made-up network of 20 channels with one hour of noise and no earthquake. The cell
counts your detections for several `k`.

In [ ]:
noise_rng = np.random.default_rng(2207)
noise_templates, noise_records, noise_truth = synth_detect.noise_channels(noise_rng, 20, 360_000, 200, smooth=4)
stacked = np.zeros(360_000)
for template, record, lag in zip(noise_templates, noise_records, noise_truth["lags"]):
    values = normalized_cc(template, record)
    stacked[:values.size - lag] += values[lag:]          # origin sample q takes the value of piece q + lag
stacked /= len(noise_templates)
spread = np.median(np.abs(stacked - np.median(stacked)))
print(f"pure noise, 1 hour, 20 channels: MAD {spread:.4f}, standard deviation {np.std(stacked):.4f} "
      f"(ratio {spread / np.std(stacked):.3f}); largest value {(stacked.max() - np.median(stacked)) / spread:.1f} "
      "MADs above the median")
for k_try in (3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0):
    times_k, _, level_k = mad_detections(stacked, 0.01, k_try, MIN_GAP_S)
    print(f"k = {k_try:4.1f}: threshold {level_k:.4f}, {len(times_k):4d} false detections in one hour")

**Explain:** From which `k` on did pure noise give no false detection, and how much margin does `K_MAD = 8` leave?

## Part 4 · Relative magnitudes (about 30 min)

A detection says when, not how big. One magnitude unit is 10 times the amplitude, so an event 10 times smaller
than the template is one unit smaller: a **relative magnitude**. The amplitude here is the **L2 norm** of a window
(the square root of its sum of squares), and the median over the channels resists one odd channel.

### M22-T4 · A relative magnitude · core · about 30 min

Write `relative_magnitude(template_windows, event_windows, template_ml)`. A channel's step is
$\log_{10}$(event norm / template norm), each window minus its mean; skip a channel where a norm is 0. Return
`template_ml` plus the median step, or `nan` when no channel is left.

Example (`template_ml = 1.0`):

- `template_windows = [[1, -1, 1, -1], [2, 0, -2, 0]]`
- `event_windows = [[10, -10, 10, -10], [0, 2, 0, -2]]`
- give 1.5 (steps 1.0 and 0.0).

Not allowed: the course package.

In [ ]:
# Before you code: an event identical to the template but 3 times larger on every channel. What is its magnitude
# relative to the template?
st.predict("M22-T4", """

""")

In [ ]:
def relative_magnitude(template_windows, event_windows, template_ml):
    """Magnitude of an event from the median over channels of log10(L2 norm ratio) against the template.

    template_windows : list of 1-D sequences of numbers, the template's window on each channel
    event_windows    : list of the same length: the event's window on the same channels (same length per channel)
    template_ml      : magnitude of the template's earthquake
    On each channel remove each window's own mean; the channel's step is log10(norm of the event window / norm of
    the template window), with the L2 norm (square root of the sum of squares). Leave out a channel where either
    window is flat (norm 0).
    returns          : template_ml + the median of the steps (a float; with an even number of channels the mean of
                       the two middle steps); nan when no channel is left
    """
    # Step 1: on each channel, both windows minus their means; their L2 norms.
    # Step 2: skip a channel with a norm of 0; else the step is log10(event norm / template norm).
    # Step 3: template_ml plus the median of the steps.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T4: write your code here")

In [ ]:
st.check("M22-T4", func=relative_magnitude)

Your function against the least-squares ratio (coefficient times norm ratio) for a made-up event 10 times larger
than the template: first with the template's waveform, then with one only partly like it.

In [ ]:
mag_rng = np.random.default_rng(5)
shapes = [synth_detect.wavelet(mag_rng, 300) for _ in range(5)]
same = [10.0 * s for s in shapes]
other = []
for s in shapes:
    w = synth_detect.wavelet(mag_rng, 300)
    mixed = 0.6 * s / np.linalg.norm(s) + 0.8 * w / np.linalg.norm(w)      # partly like the template
    other.append(10.0 * mixed * np.linalg.norm(s) / np.linalg.norm(mixed))
for label, events in (("same waveform", same), ("other waveform", other)):
    coefficients = [float(normalized_cc(s, e)[0]) for s, e in zip(shapes, events)]
    least_squares = 1.5 + np.median([np.log10(max(c, 1e-6) * np.linalg.norm(e) / np.linalg.norm(s))
                                     for c, s, e in zip(coefficients, shapes, events)])
    print(f"{label:>14s}: coefficients {np.round(coefficients, 2)}; your magnitude "
          f"{relative_magnitude(shapes, events, 1.5):.2f}, least-squares magnitude {least_squares:.2f} "
          "(template 1.5, events 10 times larger)")

**Explain:** Why does the least-squares magnitude fall for the second kind of event, while yours does not?

## Part 5 · Two templates on three hours (about 120 min)

Now two templates from your own picks: the ML 3.9 (M20) and the ML 1.5 at 02:10:52 UTC. A window starts `PRE_S`
before a pick: P on the vertical channel, S on both horizontals.

**Pick the ML 1.5** below, as in M20. Its amplitude is about 250 times smaller than that of the ML 3.9, so
switch on the band-pass.

In [ ]:
SMALL_ORIGIN = KMA_TIMES[SMALL_ID]
around_small = raw.slice(SMALL_ORIGIN - 30.0, SMALL_ORIGIN + 60.0).copy()
small_picker = st.widgets.phase_picker(around_small, origin=SMALL_ORIGIN, epicenter=(EV_LAT, EV_LON),
                                       csv_path=SMALL_PICKS, window=(-3.0, 20.0), fmin=2.0, fmax=15.0)
small_picker

**The null test.** How many detections could be chance? Reverse every record in time and scan again with the same
templates, lags and threshold. This **null test** breaks the arrival pattern on purpose, so its peaks are false
candidates.

**Watch out:** a reversed record is a useful control, not an event-free one: some waveforms and noise still
correlate. Its count is not the false-alarm rate of ordinary records (Part 7).

### M22-T5 · Template matching on three hours · core · about 120 min

Scan the three hours with both templates: the detections, their relative magnitudes, their KMA ids (within
`MATCH_S`) and the null test. Write two files to `results/` and set `tm_result`; the comments in the code cell
list every step and key. Use your own four functions: the check reruns them and recomputes your results.

In [ ]:
# Before you run: besides the two KMA events, how many detections do you expect from each template, and how big?
st.predict("M22-T5", """

""")

In [ ]:
# Pick the ML 1.5 in the picker above and press Save (or type the picks, one call per pick:
#     small_picker.set_pick("XX.STA1", "P", <seconds after 02:10:52>, <uncertainty in s>, "?", channel="HHZ")
# and then small_picker.save()). Then do these steps for each of the two templates (write small functions and call
# them for both), using your normalized_cc, mad_detections, relative_magnitude and picks_from_probability:
# Step 1 (windows): read its picks file; for each pick at a station of `prepared`, cut from `prepared` P_LEN_S
#   seconds on the channel ending in Z (P pick) or S_LEN_S seconds on both channels ending in N and E (S pick),
#   starting at the sample i nearest to pick - PRE_S. A template needs windows at 3 or more stations.
# Step 2 (lags): o = the sample nearest to the template's KMA origin time; each channel keeps its lag d = i - o.
# Step 3 (network cross-correlation): an array of zeros as long as the records; for each channel take
#   normalized_cc of its window with its whole record and add the value at q + d to element q (only where q + d
#   exists); divide by the number of channels. At its own origin the template finds itself with a value of 1
#   (self-detection): if it does not, your lags are off.
# Step 4 (detections): mad_detections on that array with K_MAD and MIN_GAP_S; a detection at t seconds is the
#   sample q = round(t / DELTA), the time T0 + q * DELTA. Keep the median, the MAD and the threshold.
# Step 5 (null test): repeat step 3 with every record reversed ([::-1]), the same windows and lags; count the peaks
#   of picks_from_probability on it at the threshold of step 4 (not a new one).
# Step 6 (magnitudes): for each detection q, relative_magnitude of the windows against the pieces of `prepared`
#   that start at q + d (leave out channels whose piece runs past the end), with the template's KMA ML.
# Step 7 (files and result):
#   results/M22_detections.csv: one row per detection of either template, sorted by template, then time, with the
#     columns template (the KMA id of the template's event), time_utc (ISO 8601: T0 + q * DELTA), network_cc,
#     magnitude and kma_event_id (the id of the KMA event whose origin time lies within MATCH_S of time_utc,
#     empty when there is none).
#   results/M22_template_matching.json with the keys dataset, band_hz ([FMIN_HZ, FMAX_HZ]), corners, taper_s,
#     pre_s, p_len_s, s_len_s, k_mad, min_gap_s, match_s, detections_csv (the path of the CSV file) and
#     templates: one entry per template with event_id, ml, origin_utc, picks_csv (the picks file it came from),
#     channels, median, mad, threshold, n_detections, null_threshold (the threshold used on the reversed
#     records) and null_n_detections. channels is a list of {"id": "<NET>.<STA>..<CHA>", "phase": "P" or "S",
#     "start_utc": the time of sample i, "lag_samples": d}, for example {"id": "XX.STA1..HHZ", "phase": "P",
#     "start_utc": "2031-04-05T06:07:08.090000Z", "lag_samples": 412} (invented values).
#   Last, tm_result = {"json": <path of the JSON file>, "ncc": normalized_cc, "detect": mad_detections,
#     "magnitude": relative_magnitude, "network": {<KMA id>: (<network cross-correlation of the records>,
#     <that of the reversed records>), ...}}: the two numpy arrays of each template (the cells after the check
#     draw them).
# YOUR CODE HERE
raise NotImplementedError("M22-T5: write your code here")

In [ ]:
st.check("M22-T5", answer=tm_result, dataset=DATASET)

Your detections and the null test:

In [ ]:
summary = json.loads(Path(tm_result["json"]).read_text(encoding="utf-8"))
found_table = pd.read_csv(summary["detections_csv"], dtype={"template": str, "kma_event_id": str},
                          keep_default_na=False)
hours = (T1 - T0) / 3600.0
for entry in summary["templates"]:
    print(f"template {entry['event_id']} (ML {entry['ml']}), {len(entry['channels'])} channels at "
          f"{len({c['id'].split('.')[1] for c in entry['channels']})} stations: MAD {entry['mad']:.4f}, threshold "
          f"{entry['threshold']:.3f}; {entry['n_detections']} detections on the records, "
          f"{entry['null_n_detections']} on the reversed records ({entry['null_n_detections'] / hours:.2f} per hour)")
shown = found_table.assign(mads_above=[
    round((row.network_cc - entry["median"]) / entry["mad"], 1)
    for row in found_table.itertuples()
    for entry in summary["templates"] if entry["event_id"] == row.template])
print(shown.round({"network_cc": 3, "magnitude": 2}).to_string(index=False))

Counts at other `k`, on the records and on the reversed records:

In [ ]:
network = tm_result["network"]
null_rows = []
for entry in summary["templates"]:
    real_cc, null_cc = network[entry["event_id"]]
    for k_try in (5.0, 6.0, 7.0, 8.0, 9.0, 10.0):
        on_real, _, level_k = mad_detections(real_cc, DELTA, k_try, MIN_GAP_S)
        on_null, _ = picks_from_probability(null_cc, DELTA, level_k, MIN_GAP_S)
        null_rows.append({"template": entry["event_id"], "k": k_try, "threshold": round(level_k, 3),
                          "records": len(on_real), "reversed": len(on_null)})
print(pd.DataFrame(null_rows).pivot_table(index="k", columns="template", values=["records", "reversed"]))

Network cross-correlations and thresholds; below each, the reversed records:

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8.0, 7.5), sharex=True)
minutes = np.arange(len(network[MAINSHOCK_ID][0])) * DELTA / 60.0
for row_axes, entry in zip((axes[:2], axes[2:]), summary["templates"]):
    real_cc, null_cc = network[entry["event_id"]]
    for ax, values, label in ((row_axes[0], real_cc, "records"), (row_axes[1], null_cc, "reversed")):
        ax.plot(minutes, values, color="k", lw=0.3)
        ax.axhline(entry["threshold"], color="tab:red", lw=0.8, ls="--")
        ax.set_ylabel(f"ML {entry['ml']}\n{label}", fontsize=8)
    row_axes[1].set_ylim(row_axes[0].get_ylim())                  # one scale for the records and the reversed
    mine = found_table[found_table["template"] == entry["event_id"]]
    row_axes[0].plot([(UTCDateTime(t) - T0) / 60.0 for t in mine["time_utc"]], mine["network_cc"], "o",
                     mfc="none", color="tab:red", ms=6)
axes[-1].set_xlabel(f"Minutes after {T0.strftime('%H:%M:%S')} UTC")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "network_cc_three_hours.png", dpi=150)

A detection missing from the KMA catalog is not automatically a new earthquake. Compare its S windows with those
of the ML 1.5 template:

In [ ]:
new_ones = found_table[(found_table["kma_event_id"] == "") & (found_table["template"] == SMALL_ID)]
small_entry = next(e for e in summary["templates"] if e["event_id"] == SMALL_ID)
s_channels = [c for c in small_entry["channels"] if c["phase"] == "S"][:6]
if len(new_ones):
    fig, axes = plt.subplots(len(s_channels), len(new_ones) + 1, figsize=(2.6 * (len(new_ones) + 1), 6.0),
                             squeeze=False, sharex=True)
    for col, start in enumerate([SMALL_ORIGIN] + [UTCDateTime(t) for t in new_ones["time_utc"]]):
        for row, ch in enumerate(s_channels):
            tr = prepared.select(id=ch["id"])[0]
            first = int(round((start - tr.stats.starttime) / DELTA)) + ch["lag_samples"]
            piece = tr.data[first - 100:first + int(S_LEN_S / DELTA) + 100]
            axes[row, col].plot(np.arange(piece.size) * DELTA - 1.0, piece / np.abs(piece).max(), color="k",
                                lw=0.5)
            axes[row, col].set_yticks([])
            if col == 0:
                axes[row, col].set_ylabel(ch["id"].split(".", 1)[1].replace("..", " "), fontsize=7)
        axes[0, col].set_title("template ML 1.5" if col == 0 else start.strftime("%H:%M:%S"), fontsize=8)
    for ax in axes[-1]:
        ax.set_xlabel("Seconds after the window start")
    fig.tight_layout()
    fig.savefig(Path(FIGURE_DIR) / "new_detections.png", dpi=150)
print(f"{len(new_ones)} detections of the ML 1.5 template are not in the KMA catalog")

**Explain:** Which of your detections are not in the KMA catalog, and which would you keep as earthquakes after
looking at their waveforms?

**Optional: explorer E11.** At which `k` does the reversed record stop giving detections?

In [ ]:
# Optional explorer E11 (not checked): your threshold and detections on the ML 1.5 template's network
# cross-correlation, and the false detections on the reversed records.
small_real, small_null = network[SMALL_ID]
st.widgets.detection_explorer(small_real, DELTA, detect=mad_detections, peaks=picks_from_probability,
                              null_cc=small_null, starttime=T0, k=K_MAD, min_gap_s=MIN_GAP_S,
                              marks={f"ML {KMA_ML[e]:.1f}": t for e, t in KMA_TIMES.items()})

## Part 6 · A deep-learning detector and an associator (about 90 min)

**PhaseNet** (M21) picks P and S at every station; **phase association** groups the picks into earthquakes. Here
the place is known. So subtract from each pick the travel time from the KMA hypocenter to its station: that is the
pick's **back-projected origin time**. Picks of one earthquake point to nearly the same time, noise picks anywhere.

### M22-T6 · An associator · core · about 60 min

Write `associate(picks, travel_times, window_s, min_stations)`. Sort the picks by back-projected time. From each
free pick (not yet in an event), take the free picks up to `window_s` later, the **association window**. Picks
from at least `min_stations` stations make an event at the median of their times.

Example (`window_s = 0.6`, `min_stations = 3`):

- picks `XX.STA1 P 12.0`, `XX.STA2 P 13.0`, `XX.STA3 S 16.4` and `XX.STA1 S 30.0`
- travel times 2.0, 3.2, 6.1 and 3.5 s
- give one event: origin 10.0 s, 3 stations, 3 picks.

Not allowed: PyOcto, GaMMA and scikit-learn.

In [ ]:
# Before you code: will the associator find, in PhaseNet's picks, the small events that your templates found?
st.predict("M22-T6", """

""")

In [ ]:
# Your direct_time of module M30 (its task 1): paste it here, unchanged. Part 6 uses it after the check.
# YOUR CODE HERE
raise NotImplementedError("M22-T6: write your code here")

In [ ]:
def associate(picks, travel_times, window_s, min_stations):
    """Group picks into events by their back-projected origin times.

    picks        : DataFrame with the columns station, phase ("P" or "S") and time_s (s after one common start);
                   do not change it
    travel_times : dict {(station, phase): predicted travel time in s}; picks without one are left out
    window_s     : s (> 0)
    min_stations : whole number (1 or more)
    A pick's back-projected time is time_s - travel_times[(station, phase)]; go through the picks sorted by it.
    From each free pick, take it and every later free pick at most window_s after it (the edge counts). If they
    come from at least min_stations different stations, they make an event and are no longer free; otherwise go
    on with the next pick.
    returns      : DataFrame with the columns origin_s (float: the median of the event's back-projected times),
                   n_stations and n_picks (whole numbers), one row per event, sorted by origin_s; an empty
                   DataFrame with these columns when there is no event
    Not allowed here: PyOcto, GaMMA, scikit-learn, hdbscan, scipy.cluster and the course package.
    """
    # Step 1: (back-projected time, station) for every pick with a travel time; sort by the time.
    # Step 2: walk through them; from each pick not yet in an event, gather the later free picks within window_s.
    # Step 3: enough stations: an event (median origin), and its picks are taken; else go on with the next pick.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T6: write your code here")

In [ ]:
st.check("M22-T6", func=associate)

Your `direct_time` gives the travel times; the cell also back-projects your ML 3.9 picks.

**Watch out:** the associator predicts direct waves only. At 14 km in kim1983 all eight stations lie inside the
crossover distances of M30 (65 km for P, 50 km for S). For a deeper aftershock a head wave may lead slightly.

In [ ]:
model = st.data.velocity_model(VELOCITY_MODEL)
travel = {}                                      # {(station, phase): seconds} from the KMA hypocenter
for row in stations.itertuples():
    for phase in ("P", "S"):
        travel[(row.station, phase)] = float(direct_time(row.epi_km, EV_DEPTH_KM, model, phase))
main_picks = m20_picks[m20_picks["station"].isin(stations["station"])]
fit = pd.DataFrame({"station": main_picks["station"], "phase": main_picks["phase"],
                    "travel_s": [travel[(s, p)] for s, p in zip(main_picks["station"], main_picks["phase"])],
                    "back_projected_s": [UTCDateTime(t) - KMA_TIMES[MAINSHOCK_ID] - travel[(s, p)]
                                         for s, p, t in zip(main_picks["station"], main_picks["phase"],
                                                            main_picks["time_utc"])]})
print(f"travel times in {VELOCITY_MODEL} from {EV_DEPTH_KM:g} km depth; your ML 3.9 picks, back-projected "
      "(s after the KMA origin):")
print(fit.round(3).to_string(index=False))
print(f"spread of the back-projected times: {fit['back_projected_s'].max() - fit['back_projected_s'].min():.2f} s")

Your associator on made-up picks: five events, a third of the picks missing, and 40 stray picks.

In [ ]:
made_up_picks, made_up_truth = synth_detect.pick_table(np.random.default_rng(31), list(stations["station"]), travel,
                                                        [600.0, 1500.0, 1520.0, 2400.0, 3300.0], jitter_s=0.3,
                                                        p_miss=0.35, n_stray=40, duration_s=3600.0)
print(f"{len(made_up_picks)} made-up picks; the five events got picks at "
      f"{[len(s) for s in made_up_truth['stations']]} stations")
for needed in (2, 3, 4, 5):
    found = associate(made_up_picks, travel, ASSOC_WINDOW_S, needed)
    true_ones = sum(any(abs(o - t) < 1.0 for t in made_up_truth["origin_s"]) for o in found["origin_s"])
    print(f"min_stations {needed}: {len(found)} events, {true_ones} of them within 1 s of a made-up origin time")

**Explain:** Why must the association window be wider than the spread of your own ML 3.9 picks above, but not much
wider?

**PhaseNet, three runs** on the raw records, with the weights of M21. Run 1 uses `PN_THRESHOLD`. Run 2 repeats it
unchanged: is the output **deterministic**? Run 3 changes one setting, `PN_THRESHOLD_CHANGED`.

In [ ]:
import torch

torch.set_num_threads(1)                         # inference on one core (shared-server rule)
try:
    import seisbench
    import seisbench.models as sbm
except ImportError as exc:
    raise ImportError(f"SeisBench cannot load its models in this environment ({exc}). Run 'sgtl doctor' in a "
                      "terminal and tell your mentor.") from None
pinned = {}
for line in st.data.repo_path("install", "checksums.txt").read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) == 3 and parts[0].startswith("phasenet/"):
        pinned[parts[0]] = parts[1]
for kind in ("json", "pt"):
    rel = f"phasenet/{PN_WEIGHTS[0]}.{kind}.v{PN_WEIGHTS[1]}"
    path = Path(seisbench.cache_model_root) / rel
    if not path.is_file():
        raise FileNotFoundError(f"{path} is missing. In a terminal: bin/sgtl-install seisbench --apply")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{rel}: SHA-256 {digest[:16]}... "
          f"{'matches the course pin' if pinned.get(rel) == digest else 'DOES NOT MATCH the pin: tell your mentor'}")
phasenet = sbm.PhaseNet.from_pretrained(PN_WEIGHTS[0], version_str=PN_WEIGHTS[1])
phasenet.eval()


def phasenet_picks(threshold):
    """PhaseNet's picks on the raw three hours (P and S at one threshold), as a table in seconds after T0."""
    output = phasenet.classify(raw, P_threshold=threshold, S_threshold=threshold)
    table = pd.DataFrame([{"station": ".".join(p.trace_id.split(".")[:2]), "phase": p.phase,
                           "time_s": float(p.peak_time - T0), "probability": float(p.peak_value)}
                          for p in output.picks], columns=["station", "phase", "time_s", "probability"])
    return table.sort_values(["time_s", "station", "phase"], ignore_index=True)


pn_first = phasenet_picks(PN_THRESHOLD)
pn_rerun = phasenet_picks(PN_THRESHOLD)
pn_changed = phasenet_picks(PN_THRESHOLD_CHANGED)
print(f"threshold {PN_THRESHOLD}: {len(pn_first)} picks; rerun identical: {pn_rerun.equals(pn_first)}")
print(f"threshold {PN_THRESHOLD_CHANGED}: {len(pn_changed)} picks")
counts = pd.concat({f"threshold {PN_THRESHOLD}": pn_first.groupby(["phase", "station"]).size(),
                    f"threshold {PN_THRESHOLD_CHANGED}": pn_changed.groupby(["phase", "station"]).size()}, axis=1)
print(counts.fillna(0).astype(int).T.to_string())
away = np.array([all(abs(T0 + t - k) > 30.0 for k in KMA_TIMES.values()) for t in pn_changed["time_s"]], dtype=bool)
far = pn_changed[(pn_changed["phase"] == "P").to_numpy() & away]
print(f"\nP picks at threshold {PN_THRESHOLD_CHANGED} more than 30 s from both KMA origin times:")
print(far.assign(time_utc=[str(T0 + t) for t in far["time_s"]])[["time_utc", "station", "probability"]]
      .round(2).to_string(index=False))

Your associator on PhaseNet's picks, and all lists side by side. Entries less than 3 s apart are one event.

In [ ]:
pn_events = {}
for label, table in (("PhaseNet " + str(PN_THRESHOLD), pn_first), ("PhaseNet " + str(PN_THRESHOLD_CHANGED),
                                                                  pn_changed)):
    found = associate(table, travel, ASSOC_WINDOW_S, ASSOC_MIN_STATIONS)
    pn_events[label] = [T0 + float(t) for t in found["origin_s"]]
    print(f"{label}: {len(found)} events")
    print(found.assign(origin_utc=[str(T0 + float(t)) for t in found["origin_s"]]).round(2).to_string(index=False))

lists = {"KMA": list(KMA_TIMES.values()),
         "templates": [UTCDateTime(t) for t in sorted(set(found_table["time_utc"]))],
         **pn_events}
merged = []
for name, times in lists.items():
    for when in times:
        for event in merged:
            if abs(event["time"] - when) < 3.0:
                event[name] = "x"
                break
        else:
            merged.append({"time": when, name: "x"})
for event in merged:
    event["coincidence"] = "x" if any(covers(start, end, event["time"] - T0) for start, end, _ in network_found) else ""
comparison = pd.DataFrame(merged).fillna("").sort_values("time", ignore_index=True)
comparison["time"] = [t.strftime("%H:%M:%S.%f")[:-4] for t in comparison["time"]]
print(comparison.to_string(index=False))

**Explain (not checked):** Which list would you give a colleague who wants every earthquake of these three hours?

## Part 7 · One candidate per event, and a validation plan (about 120 min)

Your two templates are a small **template bank**. One earthquake can match several templates. **Cross-template
de-duplication** merges such matches into one candidate and keeps the evidence. Group them with a fixed window
from each group's earliest match, so that a chain of matches cannot stretch it.

**Held-out evaluation** tests a frozen detector on records not used to tune it. You have seen these three hours
and set your threshold on them, so your protocol is a **retrospective teaching plan**, not a blind test. Each
project must reserve fresh time windows before any tuning.

**Watch out:** the 2019 results are teaching material, not project results. Finish this module before you search
the Suncheon or the 2026 Sangju records.

### M22-T7 · Merge a template bank and write its validation protocol · core · about 120 min

Write `merge_template_matches(matches, window_s)`: fixed-anchor groups in time order, each represented by its best
match. The docstring has the rules.

Example (`window_s = 1`):

- matches `[('A', 0.0, 8.0), ('B', 0.75, 9.0), ('C', 1.5, 10.0)]`
- give `[(0.75, 9.0, 'B', ('A', 'B'), 2), (1.5, 10.0, 'C', ('C',), 1)]`.

The first group does not grow to 1.5 s. Then merge your task 5 matches and write the two files that the code
comments list. Discuss the protocol with your mentor.

In [ ]:
# Before you code: should two templates that detect one earthquake make two events?
st.predict("M22-T7", """

""")

In [ ]:
def merge_template_matches(matches, window_s):
    """Combine fixed-anchor groups; retain representative, distinct template IDs and raw match counts.

    matches  : list of (template_id, time_s, score) tuples: non-empty text IDs and finite numbers, time_s in s
               after the record start; do not change the list
    window_s : grouping tolerance (s), finite and 0 or more
    Visit the matches in increasing time order. A group holds the earliest remaining match and every later match
    whose time is at most that earliest time + window_s (the edge counts); the anchor never moves as members join.
    The next remaining match starts a new group. The representative has the highest score; ties go to the
    earliest time, then to the alphabetically first ID.
    returns  : list in time order of (time_s, score, representative_id, template_ids, n_matches): the
               representative's time, score and ID; template_ids, a sorted tuple of the distinct supporting IDs;
               n_matches, the number of raw rows in the group (repeated rows of one template included); [] for an
               empty list
    Not allowed here: scipy.cluster, scikit-learn, EQcorrscan and the course package.
    """
    # YOUR CODE HERE
    raise NotImplementedError("M22-T7: write your code here")


# Write the candidate CSV and your protocol, then set bank_validation:
# 1. Input rows from task 5's CSV (found_table; unrounded), with the median and mad of each template from its JSON
#    (summary): (template, UTCDateTime(time_utc) - T0, (network_cc - template median) / template mad). Each
#    template's mad must be above 0.
# 2. results/M22_candidates.csv with the columns time_s,time_utc,score,representative,templates,n_matches: one row
#    per group of your merger at DEDUP_WINDOW_S; templates is a JSON list of the supporting IDs.
# 3. results/M22_validation_protocol.json with exactly these keys:
#    dataset; exercise_status ("retrospective_teaching_plan"); bank_version (non-empty text); template_ids (the
#    sorted IDs); score_definition ("mads_above_median"); merge_window_s; calibration_window_s and
#    evaluation_window_s ([start, end] in s after T0: CALIBRATION_WINDOW_S and EVALUATION_WINDOW_S); guard_s
#    (VALIDATION_GUARD_S); calibration_live_hours and evaluation_live_hours (for these gap-free records, window
#    seconds / 3600; a real project also subtracts invalid times); matching_json_sha256 and matches_csv_sha256
#    (SHA-256 of the bytes of task 5's JSON and CSV files, with hashlib); and seven written entries, each at
#    least 12 words in your own words, no "[write here]", plans rather than invented results:
#      threshold_freeze      what you freeze after calibration: bank, channel rules, filter, score, threshold,
#                            merge window and review rules (a new template makes a new bank version)
#      control_and_exposure  how you measure false alarms: confirmed false candidates divided by usable live
#                            hours, with counts and exposure; zero in a short control is not a zero rate
#      injection_recovery    how you insert known signals into separate noise windows, run the same windows
#                            without them, and keep failed insertions in the count; also waveforms unlike the bank
#      selection_bias        what a bank of cataloged, clear earthquakes favors and what it can miss
#      prior_exposure        which records you have already seen, and what that means for the evaluation
#      suncheon_transfer     how the Suncheon project builds its own bank and reserves its own unseen windows
#      sangju2026_transfer   the same for the 2026 Sangju project (never the 2019 teaching results)
#    The windows must stay inside the record by the guard at both ends, be at least two guards apart, and keep a
#    guard from each template's origin; the guard must be at least the largest |lag| plus the longest window plus
#    the taper.
# 4. bank_validation = {"matching_json": tm_result["json"], "candidates_csv": <candidate CSV path>,
#    "validation_json": <protocol JSON path>}
# YOUR CODE HERE
raise NotImplementedError("M22-T7: write your code here")

In [ ]:
st.check("M22-T7", func=merge_template_matches, report=bank_validation, dataset=DATASET)

**Explain:** Why is your protocol not a blind evaluation, and what must each project reserve before tuning?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M22")

**Optional:** make a third template from one of your new detections and scan again. Does it find more?

**Read more:** Gibbons and Ringdal (2006) and Peng and Zhao (2009), template matching; Schaff and Richards (2014),
relative magnitudes.

Next: M31 locates earthquakes from picks.